### 1. Import Hearings Metadata (from GovInfo)

In [ ]:
import requests, time, os, dotenv
dotenv.load_dotenv()

BASE = "https://api.govinfo.gov"
KEY = os.getenv('govinfokey')

params = {'api_key' : KEY,
          'pageSize' : 1000,
          'offsetMark' : '*'}

In [ ]:
# Step 1: Collect Package IDs
def get_pkg_ids(start = "2020-01-01T00:00:00Z", 
                end = "2025-12-31T00:00:00Z"):
    ids = []
    url = f"{BASE}/collections/CHRG/{start}/{end}"
    first = True
    while url:
        if first:
            r = requests.get(url, 
                         params=params).json()
            first = False
        
    else:
        r = requests.get(url,
                         params={"api_key": KEY}).json()
        ids += [p["packageId"] for p in r.get("packages", [])]
        url = r.get("nextPage")
        time.sleep(0.5)
    return ids

In [ ]:
test = get_pkg_ids()

In [ ]:
# Step 2: Collect Metadata
def get_metadata(pkg_id):
    r = requests.get(f"{BASE}/packages/{pkg_id}/summary",
                     params = params).json()
    return r

### 2. Explore Data (from Congress.API)

In [ ]:
# Inital Setup
import numpy as np
import pandas as pd
import requests
import json
import dotenv
import os
import yaml
import llm
import pprint
import time
import pyreadstat

from lxml import etree
from lxml import html
from bs4 import BeautifulSoup

In [ ]:
botname = 'targ'
version = '0.0'
email = 'kve5hd@virginia.edu'
useragent =f'{botname}/{version} ({email}) python-requests/{requests.__version__}'
headers = {'User-Agent':useragent}
headers

In [ ]:
root = "https://api.congress.gov//v3"
dotenv.load_dotenv()
congresskey = os.getenv('congresskey')

params = {'format': 'json',
          'api_key': congresskey}

In [ ]:
endpoint = f'/hearing'

r = requests.get(root + endpoint,
                 params=params,
                 headers=headers)
r

In [ ]:
myjson = r.json()

In [ ]:
myjson

### Mapping Committee-Agency using Hearings data

In [ ]:
from dotenv import load_dotenv
load_dotenv('.env')
KEY = os.getenv('govinfokey')

In [ ]:
BASE = "https://api.govinfo.gov"
results = []
offset = "*"
page = 0

In [ ]:
r = requests.post(f"{BASE}/search?api_key={KEY}", json={
    "query": "collection:CHRG",
    "pageSize": 3,
    "offsetMark": "*"
})

print(r.json().get("count"))

In [ ]:
count = 0
for congress in range(113,119):
    r = requests.post(f"{BASE}/search?api_key={KEY}", json={
        "query": f"collection:CHRG congress:{congress}",
        "pageSize": 1,
        "offsetMark": "*"
    })
    c = r.json().get("count")
    print(congress, c)
    count += c
    
print(count)

In [ ]:
results_all = []

for congress in range(113, 119):
    offset = "*"
    page = 0
    while True:
        r = requests.post(f"{BASE}/search?api_key={KEY}", json={
            "query": f"collection:CHRG congress:{congress}",
            "pageSize": 100,
            "offsetMark": offset
        })
        data = r.json()
        if not data.get("results"):
            break
        for item in data["results"]:
            results_all.append({
                "packageId": item["packageId"],
                "resultLink": item["resultLink"]
            })
        offset = data["offsetMark"]
        if len(data["results"]) < 100:
            break
        time.sleep(0.5)
    print(f"Congress {congress}: {len(results_all)} total")

print(f"\nTotal hearings: {len(results_all)}")


In [ ]:
details_all = []
for i, item in enumerate(results_all):
    try:
        r = requests.get(item["resultLink"] + f"?api_key={KEY}")
        d = r.json()

        committees = d.get("committees", [])
        agencies = d.get("agencies", [])

        details_all.append({
            "packageId": item["packageId"],
            "congress": d.get("congress"),
            "date": d.get("dateIssued"),
            "is_appropriation": d.get("isAppropriation"),
            "committees": committees,
            "agencies": agencies,
            "n_agencies": len(agencies),
            "committee_names": [c.get("committeeName") for c in committees],
            "committee_ids": [c.get("authorityId") for c in committees],
            "chambers": [c.get("chamber") for c in committees],
            "agency_names": [a.get("name") for a in agencies],
        })
    except Exception as e:
        print(f"Error on {item['packageId']}: {e}")

    if (i + 1) % 500 == 0:
        print(f"Processed {i+1}/{len(results_all)}")
    time.sleep(0.5)

govinfo_all = pd.DataFrame(details_all)
print(f"Shape: {govinfo_all.shape}")

In [ ]:
# How many non-appropriation hearings have agency tags?
govinfo_all['has_agency'] = govinfo_all['n_agencies'] > 0
print(govinfo_all.groupby('is_appropriation')['has_agency'].value_counts())

In [ ]:
import json, pandas as pd

with open("../data/hearings_govinfo_113_118.json") as f:
    govinfo_all = pd.DataFrame(json.load(f))

govinfo_all['has_agency'] = govinfo_all['n_agencies'] > 0
print(f"Loaded {len(govinfo_all)} rows")
govinfo_all.head()